# 3.3 Synthetic Regression Data

Mainly helpful for evaluating properties of our learning algorithms, and to confirm implementations work as expected.
- e.g. Checking if a model can recover the correct parameters

In [8]:
%matplotlib inline
import random
import torch
from d2l import torch as d2l

### 3.3.1: Generating the Dataset

Generate 1000 examples with 2-dimensional features drawn from a standard normal distribution.

Resulting design matrix $\bold{X}$ belong to $\mathbb{R}^{1000\times2}$. We generate each label by applying a ground truth lienar funciton, corrupting them via additive noise $\epsilon$, drawn independently for each example:

$$
\displaystyle
\bold{y}=\bold{X}\bold{w}+b+\epsilon
$$

Assume $\epsilon$ is drawn from a normal distribution with $\mu=0, \sigma=0.01$.

In [9]:
class SyntheticRegressionData(d2l.DataModule): #@save
    """Synthetic data for linear regression."""
    def __init__(self, w, b, noise=0.01, num_train=1000, num_val=1000,batch_size=32):
        super().__init__()
        self.save_hyperparameters()
        n = num_train+num_val
        self.X = torch.randn(n, len(w))
        noise = torch.randn(n,1) * noise
        self.y = torch.matmul(self.X,w.reshape((-1,1))) + b + noise

In [10]:
data = SyntheticRegressionData(w=torch.tensor([2,-3.4]),b=4.2)
print('features:', data.X[0], '\nlabel:', data.y[0])

features: tensor([0.7014, 0.9278]) 
label: tensor([2.4423])


### 3.3.2: Reading the Dataset

Training machine learning models often require multiple passes over a dataset, grabbing a minibatch of examples at a time.
This data is then used to update the model. To illustrate how this works, we implement the `get_dataloader` method, reigstering it in the `SyntheticRegressionData` class via `add_to_class`. It takes a batch size, a matrix of features, and a vector of labels, and generates minibatches of size `batch_size`. As such, each minibatch consists of a tuple of features and labels.

Note taht we need to be mindful of whether we are in training or validation mode: in the former, we want data to be read in a random order, whereas for the latter, being able to read data in a pre-defined order may be important for debugging purposes.

In [11]:
@d2l.add_to_class(SyntheticRegressionData)
def get_dataloader(self, train):
    if train:
        indices = list(range(0, self.num_train))
        # The examples are read in random order
        random.shuffle(indices)
    else:
        indices = list(range(self.num_train, self.num_train + self.num_val))
    for i in range(0, len(indices), self.batch_size): # Read `self.batch_size` examples at a time
        # Take self.batch_size = 32 values from the shuffled list of indexes. 
        batch_indices = torch.tensor(indices[i:i+self.batch_size]) 
        yield self.X[batch_indices], self.y[batch_indices]

Each minibatch of features gives us the size and dimensionality of input features. Our minibatch of labels will have a matching shape give by `batch_size`.

In [12]:
X,y = next(iter(data.train_dataloader()))
print('X shape:', X.shape, '\ny shape:', y.shape)

X shape: torch.Size([32, 2]) 
y shape: torch.Size([32, 1])


Note that `get_dataloader()` was previously undefined in `Exercises 3.2.ipynb`.

`SyntheticRegressionData.train_dataloader() -> SyntheticRegressionData.get_dataloader(train=True) -> return shuffled data generator`

## Aside: Python Generators:

- Generators are a special type of function that returns an iterator, allowing for the creation of sequences on demand rather than storing the entire sequence in memory at once
- This 'lazy evaluation' makes generators highly memory-efficient, especially when it comes with large or near-infinite datasets.
    - Saves space by only ever storing one (or a few) elements in memory at a time, evaluating values on-the-fly rather than storing all values of a list at all times.
- `yield`: A generator's replacement for `return`. When encoutnered, the function pauses its execution, returns yielded value, and saves internal state.
- Generator functions can be automatically iteratied over using `for` loops or by repeatedly calling the built-in `next()` function
- "Generator Objects": an object that stores internal state and delays execution until `next()` is called on the object. Once called, it resumes from where it last yielded a value.
- When there are no more values to yield, a `StopIteration` exception is raised, signaling completion.

### A simple generator:

In [13]:
def simple_generator():
    yield 1
    yield 2
    yield 3

gen=simple_generator()

# Iterating using `next()`
print(next(gen)) # 1
print(next(gen)) # 2
print(next(gen)) # 3

# print(next(gen)) # Trying to get another value will raise a StopIterationException

print("----------")
# Iterating using a for loop
for value in simple_generator():
    print(value)
    """
    Output:
    1
    2
    3
    """

1
2
3
----------
1
2
3


While iteration over a dataset is good for didactic (educational) purposes, it can be ineffiecient in practice. For example, it requires we load all data in memory and we perform lots of random memory accesses. 

The built-in iterators in a deep learning framework are considerably more efficient and can deal with sources such as files, streams, and data generated or processed on the fly.

## 3.3.3: Concise Implementation of the Data Loader

In [ ]:
@d2l.add_to_class(d2l.DataModule) #@save
def get_tensorloader(self, tensors, train, indices=slice(0,None)):
    """Use the existing Pytorch generator via its API to optimize our data"""
    tensors = tuple(a[indices] for a in tensors)
    dataset = torch.utils.data.TensorDataset(*tensors)
    return torch.utils.data.DataLoader(dataset, self.batch_size, shuffle=train)

@d2l.add_to_class(SyntheticRegressionData) #@save
def get_dataloader(self, train):
    i = slice(0, self.num_train) if train else slice(self.num_train, None)
    return self.get_tensorloader((self.X, self.y), train, i)

The new dataloader behaves just like the previous one, except it is more efficient and has more functionality.

Note: API = "Application Programming Interface"

In [15]:
X,y = next(iter(data.train_dataloader()))
print('X shape:', X.shape, '\ny shape:', y.shape)

X shape: torch.Size([32, 2]) 
y shape: torch.Size([32, 1])


In [16]:
# Example of a new feature in the optimized dataloader: __len__ support
len(data.train_dataloader())

32

# Summary

- Data loaders are a convenient way of abstracting out the process of loading and manipulating data. This way, the same machine learning algorithm is capable of processing many different types and sources of data without the need for modification.
- Data loaders can be composed (e.g. include a pipeline of steps) and thus can be used to describe an entire data processing pipeline
- The two-dimensional linear model is about the simplest model we may encounter. **This allows us to test the accuracy of regression models without worrying about having insufficient amounts of data or undetermined system of equations.**
    - We will put this to good use in the next section.

# Exercises:

1. What will happen if the number of examples cannot be divided by the batch size. How would you change this behavior by specifying a different argument by using the framework's API?

If the number of examples cannot be divided by the batch size, the last batch will include a number of examples equal to the remainder. We can change this behavior using `torch.utils.DataLoader`'s `drop_last` argument. 

`drop_last=False` (default) $\rightarrow$ DataLoader will yield a final batch that contains remaining samples.

`drop_last=True`            $\rightarrow$ DataLoader will discard the last incomplete batch.

2. Suppose that we want to generate a huge dataset, where both the size of the parameter vector `w` and the number of examples `num_examples` are large.
    1. What happens if we cannot hold all data in memory?

    If we cannot hold all data in memory, then we must use a generator to store the data. Otherwise, we will encounter Out-Of-Memory errors (OOMs) when attempting to load the data.

    1. How would you shuffle the data if it is held on disk? Your task is to design an *efficient* algorithm that does not require too many reads or writes. Hint: pseudorandom permutation generators allow you to design a reshuffle without the need to store the permutation table explicity (Naor and Reingold, 1999).

    Iterate through the array from last element to second element. At each iteration, select a random element from the unshuffled portion of the array (`[0:current element]`) and swap it with the current element. This is known as the Fisher-Yates shuffle.

1. Implement a data generator that produces new data on the fly, every time the iterator is called.

In [17]:
def on_the_fly_generator():
    for i in range(20):
        yield i*i

for res in on_the_fly_generator():
    print(res)

0
1
4
9
16
25
36
49
64
81
100
121
144
169
196
225
256
289
324
361


4. How would you design a random data generator that generates *the same* data each time it is called?

To generate the same result each time the generator is called, we can set a `random_state` or `seed` to the generator's randomness. This way, all calls of the generator will produce a value that is random and consistent between calls.